# Short Text Datasets for EVT-MinHash Near-Duplicate Detection Evaluation

This notebook demonstrates the dataset preparation for evaluating EVT-MinHash near-duplicate detection on short text documents.

## What this artifact does:

1. **Loads pre-collected short text datasets** from HuggingFace Hub
2. **Standardizes the data format** to a common schema for experiment evaluation
3. **Computes text statistics** (word counts, k-shingle counts) for MinHash evaluation

## Datasets included:

- **cardiffnlp/tweet_eval (sentiment)**: 10,000 tweets (avg 19.2 words)
- **cardiffnlp/tweet_eval (emoji)**: 10,000 tweets (avg 11.8 words)  
- **fancyzhx/ag_news**: 19,951 news headlines (avg 38.7 words)

All datasets contain documents with 10-100 words, suitable for near-duplicate detection evaluation.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru - NOT pre-installed on Colab, always install
_pip('loguru')

# Core packages (pre-installed on Colab, install locally to match Colab env)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2')

print('Dependencies installed successfully!')

In [ ]:
from loguru import logger
from pathlib import Path
import json
import sys

# Setup logging
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

print('Imports loaded successfully!')

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-1ba2b3-why-extreme-value-theory-fails-for-minha/main/round-1/dataset-1/demo/mini_demo_data.json"

import json, os

def load_data():
    """Load demo data from GitHub URL with local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception as e:
        print(f"GitHub load failed: {e}")
    
    # Fallback to local file
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json")

print('Data loading helper defined!')

In [ ]:
# Load the demo data
data = load_data()

print(f"Loaded data with {data['metadata']['total_examples']} examples from {data['metadata']['total_datasets']} datasets")
print(f"Sources: {', '.join(data['metadata']['source_datasets'])}")

## Understanding the Data Structure

The demo data is already in the standardized format. Let's examine its structure:

- **metadata**: Information about the dataset collection
- **datasets**: List of datasets, each containing:
  - **dataset**: Name of the source dataset
  - **examples**: List of text examples with:
    - **input**: The text document
    - **output**: Document identifier (doc_id)
    - **metadata_source**: Source dataset name
    - **metadata_word_count**: Number of words in the document
    - **metadata_shingle_count**: Number of k=3 shingles (for MinHash)
    - **metadata_doc_id**: Unique document ID

In [ ]:
# Explore the data structure
print("=== Data Structure ===")
print(f"Number of datasets: {len(data['datasets'])}")
print()

for dataset in data['datasets']:
    print(f"Dataset: {dataset['dataset']}")
    print(f"  Number of examples: {len(dataset['examples'])}")
    print()
    
    # Show first example
    if len(dataset['examples']) > 0:
        ex = dataset['examples'][0]
        print(f"  Example fields: {list(ex.keys())}")
        print(f"  Sample text: {ex['input'][:80]}...")
        print(f"  Word count: {ex['metadata_word_count']}")
        print(f"  Shingle count (k=3): {ex['metadata_shingle_count']}")
    print()

## Original Script: Data Conversion Process

The original `data.py` script converts raw collected data (`data_out.json`) to the standardized format (`full_data_out.json`).

### What the script does:

1. **Loads raw data** from `data_out.json` (collected from HuggingFace)
2. **Groups documents by source** dataset
3. **Creates standardized examples** with required schema:
   - `input`: document text
   - `output`: doc_id (document identifier)
   - `metadata_*`: additional metadata fields
4. **Saves to full_data_out.json** in standardized format

Since we're using pre-converted demo data, we'll show the key processing steps.

In [ ]:
## Demonstrate the data conversion logic

def convert_to_standard_schema(raw_data):
    """
    Convert raw collected data to standardized schema.
    This is the core logic from data.py
    """
    # Group documents by source dataset
    datasets_dict = {}
    for doc in raw_data["documents"]:
        source = doc["source"]
        if source not in datasets_dict:
            datasets_dict[source] = []
        
        # Create example in required format
        example = {
            "input": doc["text"],
            "output": doc["doc_id"],  # Use doc_id as output identifier
            "metadata_source": source,
            "metadata_word_count": doc["length"]["words"],
            "metadata_shingle_count": doc["length"]["shingles_k3"],
            "metadata_doc_id": doc["doc_id"],
        }
        datasets_dict[source].append(example)
    
    # Convert to required schema format
    datasets_list = []
    for source, examples in datasets_dict.items():
        datasets_list.append({
            "dataset": source,
            "examples": examples
        })
    
    # Create output
    output = {
        "metadata": {
            "total_datasets": len(datasets_list),
            "total_examples": sum(len(d["examples"]) for d in datasets_list),
            "source_datasets": list(datasets_dict.keys()),
        },
        "datasets": datasets_list
    }
    
    return output

print('Conversion function defined!')
print('This replicates the core logic from data.py')

## Text Statistics Analysis

For near-duplicate detection evaluation, we need to understand the text characteristics:

- **Word count**: Document length in words
- **Shingle count (k=3)**: Number of 3-word contiguous sequences
  - Used for MinHash LSH (Locality-Sensitive Hashing)
  - More shingles = more distinctive signature

Let's analyze the statistics of our demo dataset.

In [ ]:
import matplotlib.pyplot as plt

# Collect statistics from all examples
word_counts = []
shingle_counts = []
text_lengths = []

for dataset in data['datasets']:
    for example in dataset['examples']:
        word_counts.append(example['metadata_word_count'])
        shingle_counts.append(example['metadata_shingle_count'])
        text_lengths.append(len(example['input']))

print("=== Text Statistics ===")
print(f"Total examples analyzed: {len(word_counts)}")
print()
print(f"Word count - Min: {min(word_counts)}, Max: {max(word_counts)}, Avg: {sum(word_counts)/len(word_counts):.1f}")
print(f"Shingle count - Min: {min(shingle_counts)}, Max: {max(shingle_counts)}, Avg: {sum(shingle_counts)/len(shingle_counts):.1f}")
print(f"Text length (chars) - Min: {min(text_lengths)}, Max: {max(text_lengths)}, Avg: {sum(text_lengths)/len(text_lengths):.1f}")
print()

# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Word count distribution
axes[0].hist(word_counts, bins=10, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Word Counts')
axes[0].axvline(sum(word_counts)/len(word_counts), color='red', linestyle='--', label='Mean')
axes[0].legend()

# Shingle count distribution
axes[1].hist(shingle_counts, bins=10, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Shingle Count (k=3)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Shingle Counts')
axes[1].axvline(sum(shingle_counts)/len(shingle_counts), color='red', linestyle='--', label='Mean')
axes[1].legend()

# Text length distribution
axes[2].hist(text_lengths, bins=10, edgecolor='black', alpha=0.7, color='green')
axes[2].set_xlabel('Text Length (characters)')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Distribution of Text Lengths')
axes[2].axvline(sum(text_lengths)/len(text_lengths), color='red', linestyle='--', label='Mean')
axes[2].legend()

plt.tight_layout()
plt.show()

## Preparing for Near-Duplicate Detection

The standardized dataset is now ready for near-duplicate detection evaluation.

### Next steps for EVT-MinHash evaluation:

1. **Generate near-duplicates**: Create synthetic near-duplicates by:
   - Adding/removing words
   - Reordering sentences
   - Substituting synonyms

2. **Compute MinHash signatures**: 
   - Generate k-shingles (k=3 in our metadata)
   - Compute MinHash signature for each document

3. **Evaluate detection**:
   - Use EVT (Extreme Value Theory) to set threshold
   - Measure precision/recall for near-duplicate pairs

The demo data provides a small, manageable dataset to test these steps.

In [ ]:
# Display sample documents from the dataset
print("=== Sample Documents ===")
print()

for dataset in data['datasets']:
    print(f"Dataset: {dataset['dataset']}")
    print("-" * 60)
    
    for i, example in enumerate(dataset['examples'][:3]):  # Show first 3
        print(f"\nExample {i+1}:")
        print(f"  ID: {example['metadata_doc_id']}")
        print(f"  Text: {example['input']}")
        print(f"  Words: {example['metadata_word_count']}")
        print(f"  Shingles (k=3): {example['metadata_shingle_count']}")
    
    print()
    print()

## Summary

This notebook demonstrated the dataset preparation process for EVT-MinHash near-duplicate detection evaluation.

### Key outputs:

1. **Standardized dataset format** with input text, output ID, and metadata
2. **Text statistics** showing document characteristics (word counts, shingle counts)
3. **Visualization** of text property distributions

### Files generated by the original script:

- `full_data_out.json`: Complete dataset (39,951 documents)
- `mini_data_out.json`: 3 examples per dataset for testing
- `preview_data_out.json`: Truncated text for quick inspection

### For the full evaluation:

Use the complete dataset from the repository to run the near-duplicate detection experiments and evaluate EVT-MinHash performance on short text documents.